# Lab 0 Task 0.2.1 — Transfer Learning from ImageNet
In this section, we use AlexNet on CIFAR-10 in two settings:
1. Fine-tuning the whole network
2. Feature extraction with pretrained weights

## Setup & Imports

In [1]:
import warnings
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision.datasets import CIFAR10
from torchvision.models import alexnet, AlexNet_Weights
from utils import make_loaders, evaluate, fit

# Suppress NumPy 2.4 VisibleDeprecationWarning triggered inside torchvision
warnings.filterwarnings("ignore", category=np.exceptions.VisibleDeprecationWarning)

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Make results reproducible
torch.manual_seed(1)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(1)

2.11.0+cu130
True
NVIDIA GeForce RTX 2080 Ti
Using device: cuda


## Notebook parameters

In [3]:
LOG_WANDB = True  # Notebook level parameter for enabling/disabling wandb logging
NUM_WORKERS = 8
PIN_MEMORY = True

## Load CIFAR-10

We apply ImageNet normalization and resize all images to 224x224 so they match AlexNet's expected input.

In [4]:
IMG_SIZE = 224
BATCH_SIZE = 256
VAL_FRACTION = 0.2  # 20% of training set used for validation

imagenet_mean = (0.485, 0.456, 0.406)
imagenet_std  = (0.229, 0.224, 0.225)

transform_alexnet = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

train_set = CIFAR10(root="../data", train=True,  download=True, transform=transform_alexnet)
test_set  = CIFAR10(root="../data", train=False, download=True, transform=transform_alexnet)

classes: list[str] = train_set.classes
print("Classes:", classes)

loader_kwargs = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

train_loader, val_loader, test_loader = make_loaders(
    train_set, test_set,
    loader_kwargs=loader_kwargs,
    val_fraction=VAL_FRACTION,
)

Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
Dataset split — train: 40,000  val: 10,000  test: 10,000


## AlexNet model
We replace the last classifier layer so the output dimension becomes 10.

In [5]:
def build_alexnet(pretrained=False, freeze_features=False):
    weights = AlexNet_Weights.DEFAULT if pretrained else None
    model = alexnet(weights=weights)

    in_features = model.classifier[6].in_features
    model.classifier[6] = nn.Linear(in_features, 10)

    if freeze_features:
        for p in model.features.parameters():
            p.requires_grad = False
        for layer in model.classifier[:-1]:
            for p in layer.parameters():
                p.requires_grad = False

    return model

## Experiment A — Fine-tuning AlexNet on CIFAR-10
Here we train the whole network end-to-end.

In [6]:
NUM_EPOCHS = 50
LEARNING_RATE = 1e-4

model_ft = build_alexnet(pretrained=True, freeze_features=False).to(device)
optimizer_ft = optim.Adam(model_ft.parameters(), lr=LEARNING_RATE)

wandb_kwargs = dict(
    entity="d7047e-group12",
    project="Lab0",
    name="AlexNet Fine-tuning",
    tags=["Task 0.2.1", "Fine-tuning"],
    config={"pretrained": True, "freeze_features": False, "optimizer": "Adam",
            "lr": LEARNING_RATE, "epochs": NUM_EPOCHS, "batch_size": BATCH_SIZE},
)

history_ft = fit(
    model=model_ft,
    optimizer=optimizer_ft,
    criterion=nn.CrossEntropyLoss(),
    train_loader=train_loader,
    val_loader=val_loader,
    wandb_kwargs=wandb_kwargs,
    num_epochs=NUM_EPOCHS,
    log=LOG_WANDB,
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: hamid-sabeti (d7047e-group12) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch  1/50 | train loss 0.6203, train acc 78.48% | val loss 0.4026, val acc 85.88%
Epoch  2/50 | train loss 0.3105, train acc 89.21% | val loss 0.3522, val acc 88.09%
Epoch  3/50 | train loss 0.2159, train acc 92.54% | val loss 0.3710, val acc 87.80%
Epoch  4/50 | train loss 0.1466, train acc 94.87% | val loss 0.3311, val acc 89.56%
Epoch  5/50 | train loss 0.1055, train acc 96.27% | val loss 0.3084, val acc 90.51%
Epoch  6/50 | train loss 0.0805, train acc 97.16% | val loss 0.3304, val acc 90.39%
Epoch  7/50 | train loss 0.0567, train acc 98.00% | val loss 0.3347, val acc 90.87%
Epoch  8/50 | train loss 0.0540, train acc 98.12% | val loss 0.3220, val acc 90.92%
Epoch  9/50 | train loss 0.0439, train acc 98.50% | val loss 0.3502, val acc 90.50%
Epoch 10/50 | train loss 0.0337, train acc 98.83% | val loss 0.3540, val acc 90.55%
Epoch 11/50 | train loss 0.0309, train acc 98.97% | val loss 0.3440, val acc 91.04%
Epoch 12/50 | train loss 0.0275, train acc 99.09% | val loss 0.3618, val acc

Training Accuracy,▁▅▆▆▇▇▇█████████████████████████████████
Training Loss,█▆▄▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Validation Accuracy,▁▄▃▆▇▇▇▇▇▇██▇▇▇▇█▇█▇▇████▇█▇▇▆▇▇██▇██▇▇▇
Validation Loss,▅▃▄▂▁▂▂▃▃▃▅▃▃▅▅▄▅▃▅▅▄▄▅▅▅▆▆▆▆█▇▆▆▅▆▆▇▆█▇
Training Accuracy,99.7225
Training Loss,0.00834
Validation Accuracy,91.07
Validation Loss,0.44851



Restored best weights (val loss 0.3084)


In [7]:
_, test_acc = evaluate(model=model_ft, test_loader=test_loader,
                       criterion=nn.CrossEntropyLoss(), label="AlexNet Fine-tuning")

[AlexNet Fine-tuning] Test loss: 0.3129 | Test acc: 90.35%


## Experiment B — Feature extraction with pretrained AlexNet
Here we freeze the backbone and train only the final classifier layer.

In [8]:
NUM_EPOCHS = 50
LEARNING_RATE = 1e-3

model_fe = build_alexnet(pretrained=True, freeze_features=True).to(device)
optimizer_fe = optim.Adam([p for p in model_fe.parameters() if p.requires_grad], lr=LEARNING_RATE)

wandb_kwargs = dict(
    entity="d7047e-group12",
    project="Lab0",
    name="AlexNet Feature Extraction",
    tags=["Task 0.2.1", "Feature Extraction"],
    config={"pretrained": True, "freeze_features": True, "optimizer": "Adam",
            "lr": LEARNING_RATE, "epochs": NUM_EPOCHS, "batch_size": BATCH_SIZE},
)

history_fe = fit(
    model=model_fe,
    optimizer=optimizer_fe,
    criterion=nn.CrossEntropyLoss(),
    train_loader=train_loader,
    val_loader=val_loader,
    wandb_kwargs=wandb_kwargs,
    num_epochs=NUM_EPOCHS,
    log=LOG_WANDB,
)

Epoch  1/50 | train loss 0.7580, train acc 73.60% | val loss 0.5779, val acc 80.09%
Epoch  2/50 | train loss 0.6036, train acc 78.78% | val loss 0.5246, val acc 81.83%
Epoch  3/50 | train loss 0.5657, train acc 80.15% | val loss 0.4982, val acc 82.77%
Epoch  4/50 | train loss 0.5483, train acc 80.85% | val loss 0.5086, val acc 81.98%
Epoch  5/50 | train loss 0.5335, train acc 81.02% | val loss 0.5050, val acc 82.55%
Epoch  6/50 | train loss 0.5230, train acc 81.35% | val loss 0.4968, val acc 82.83%
Epoch  7/50 | train loss 0.5134, train acc 81.69% | val loss 0.4869, val acc 83.38%
Epoch  8/50 | train loss 0.5135, train acc 81.80% | val loss 0.4834, val acc 82.89%
Epoch  9/50 | train loss 0.5090, train acc 81.93% | val loss 0.4912, val acc 83.31%
Epoch 10/50 | train loss 0.5019, train acc 82.23% | val loss 0.4825, val acc 83.43%
Epoch 11/50 | train loss 0.4937, train acc 82.14% | val loss 0.4814, val acc 83.48%
Epoch 12/50 | train loss 0.4931, train acc 82.36% | val loss 0.4887, val acc

Training Accuracy,▁▅▆▆▆▇▇▇▇▇▇▇▇▇▇█▇▇█▇████████████████████
Training Loss,█▆▅▅▄▄▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▂▂▂▁▁▂▂▁▁▁▁▁▁▁▁
Validation Accuracy,▁▅▇▅▆█▇████▇▆▆█▆▇▇▇▇█▇▇▇▆▅▇▇▇▅▇▇▇▆▆█▄▆▇▇
Validation Loss,█▄▂▃▃▁▂▁▁▂▁▂▁▁▂▂▂▃▃▂▂▃▂▂▃▄▂▂▂▃▃▃▃▃▃▂▅▃▃▃
Training Accuracy,83.0725
Training Loss,0.46971
Validation Accuracy,83.04
Validation Loss,0.50756



Restored best weights (val loss 0.4807)


In [9]:
_, test_acc = evaluate(model=model_fe, test_loader=test_loader,
                       criterion=nn.CrossEntropyLoss(), label="AlexNet Feature Extraction")

[AlexNet Feature Extraction] Test loss: 0.4782 | Test acc: 83.06%


## Brief comparison
Fine-tuning updates the whole AlexNet, so the model can adapt to CIFAR-10 more strongly.
Feature extraction keeps pretrained layers frozen, so only the new classifier learns.
That is why fine-tuning usually performs better when the target dataset is different from ImageNet.

In [10]:
print("===== Final Comparison (Accuracy) =====")
print(f"Fine-tuning     best val acc: {max(history_ft['val_acc']):.2f}%")
print(f"Feature extract best val acc: {max(history_fe['val_acc']):.2f}%")

===== Final Comparison (Accuracy) =====
Fine-tuning     best val acc: 91.50%
Feature extract best val acc: 83.48%
